# CPHY HDF5 Acquisition Generator + Dask Explorer

Generate many **OpenTelemetry-framed** dense acquisition windows as HDF5 for CPHY
telemetry, land them on RustFS (`s3://cyberphy/…`), and explore with **Dask**
(multi-file `dask.array` over object-store parts) + **Holoviews** out-of-core.

| Layer | Content |
|-------|---------|
| **A — Dataset plane** | Many contiguous `.h5` files under a product prefix (Iceberg data-file keys) |
| **B — Metadata plane** | Pointer-table *preview* only (no Iceberg write yet) |
| **C — Compute** | One Dask task per part (S3 GET + hyperslab); reductions on workers |

**Idempotent by default:** **Run All** reuses existing full-size parts under the product
prefix (no re-write). Set `FORCE_REGENERATE = True` only when you want a fresh run.

**Dask contract:** the notebook client never holds the full Values plane. Workers
fetch part objects from S3 and return only reduced blocks / tiny viz slabs. Watch
the Dask dashboard while running the compute cells.

**Profiles**

| Profile | Geometry | Approx Values size |
|---------|----------|--------------------|
| `lab` (default) | 5001 × 10000 × 108 parts | **~10 GiB** (RAID) |
| `lab_tiny` | 256 × 500 × 24 parts | ~6 MiB smoke |
| `airgap_2tb` | 5001 × 10000 × 21500 parts | **~2 TiB** |

Copy this notebook to your home before editing:
```python
import shutil; shutil.copy('/root/sample-notebooks/HDF5_CPHY_Acquisition_Generator.ipynb', '/root/')
```


In [ ]:
# --- User Configuration ---
import os
# Prefer converge/JupyterHub env via cluster_env
import sys
for _p in ("/root/sample-notebooks", "/app", "/root"):
    if _p not in sys.path:
        sys.path.insert(0, _p)
try:
    from cluster_env import load_cluster_config
    _CFG = load_cluster_config()
    print(_CFG.summary())
    # Align legacy names used below
    os.environ.setdefault("S3_BUCKET", _CFG.s3_bucket)
    if _CFG.s3_endpoint:
        os.environ.setdefault("S3_ENDPOINT", _CFG.s3_endpoint)
    os.environ.setdefault("AWS_REGION", _CFG.s3_region)
    os.environ.setdefault("DASK_SCHEDULER_ADDRESS", _CFG.dask_scheduler)
except Exception as _e:
    print("cluster_env optional:", _e)

from datetime import datetime, timezone

# Profile: "lab" | "lab_tiny" | "airgap_2tb"
PROFILE = os.getenv("HDF5_PROFILE", "lab")

# Time unit for Timestamps dataset: "ns" (OTel default) or "us" (microseconds)
TIME_UNIT = os.getenv("HDF5_TIME_UNIT", "ns")

# Contiguous storage (Iceberg/kerchunk-friendly). Set False only for chunk experiments.
CONTIGUOUS = True

# Idempotency: reuse existing full-size parts under the product object prefix (default).
FORCE_REGENERATE = os.getenv("HDF5_FORCE_REGENERATE", "0") in ("1", "true", "True", "yes")

# Optional overrides (None = use profile defaults)
N_PARTS = None          # e.g. 8 for a quicker lab run
N_SERIES = None
N_TIME = None

# S3 (JupyterHub injects these for zarf:local)
BUCKET = os.getenv("S3_BUCKET", "cyberphy")
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://127.0.0.1:9010")
S3_REGION = os.getenv("AWS_REGION", os.getenv("S3_REGION", "us-east-1"))
OUT = f"s3://{BUCKET}/"

# Dask
DASK_SCHEDULER = os.getenv(
    "DASK_SCHEDULER_ADDRESS",
    os.getenv("DASK_SCHEDULER", "tcp://cybersec-dask-scheduler.dask.svc.cluster.local:8786"),
)
USE_DASK = os.getenv("USE_DASK", "1") not in ("0", "false", "False")
# Desired worker count (0 = do not scale; only report). Multi-core lab: 8; 2 TiB: 16–32.
TARGET_WORKERS = int(os.getenv("DASK_TARGET_WORKERS", "8"))
# Wait up to this many seconds for TARGET_WORKERS to join
WORKER_WAIT_S = int(os.getenv("DASK_WORKER_WAIT_S", "120"))

# Worker-side read strides (applied inside each part task — less bytes off S3/HDF5)
# 1 = full resolution. Increase for faster first-pass analytics on lab/airgap.
SERIES_STRIDE = int(os.getenv("HDF5_SERIES_STRIDE", "1"))
TIME_STRIDE = int(os.getenv("HDF5_TIME_STRIDE", "1"))
# Cap how many parts enter the Dask array (None = all inventory parts)
MAX_PARTS = None  # e.g. 16 for a quicker dashboard demo

# Visualization (client only receives the final small slab)
HEATMAP_SERIES_STRIDE = 1
VIZ_MAX_SERIES = 256
VIZ_MAX_TIME = 2000
VIZ_MAX_POINTS = 2_000_000

print(f"PROFILE={PROFILE}  TIME_UNIT={TIME_UNIT}  BUCKET={BUCKET}")
print(f"FORCE_REGENERATE={FORCE_REGENERATE}  (False → reuse existing parts on Run All)")
print(f"S3_ENDPOINT={S3_ENDPOINT or '(AWS default)'}")
print(f"DASK={DASK_SCHEDULER if USE_DASK else 'disabled'}")
print(f"READ strides series={SERIES_STRIDE} time={TIME_STRIDE}  MAX_PARTS={MAX_PARTS}")
print(f"TARGET_WORKERS={TARGET_WORKERS}  (set DASK_TARGET_WORKERS=0 to skip scale wait)")


In [ ]:
# Imports + path to generate_hdf5
import json
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve generate_hdf5.py.
# Prefer ConfigMap/home copies (live-updatable) over image-baked /app so ensure_parts
# and other fixes land without a Jupyter image rebuild. expanduser() required for ~/.
_CANDIDATES = [
    Path("~/sample-notebooks/generate_hdf5.py").expanduser(),  # ConfigMap mount
    Path("/root/sample-notebooks/generate_hdf5.py"),
    Path.cwd() / "generate_hdf5.py",
    Path.cwd() / "zarf" / "scripts" / "generate_hdf5.py",
    Path("/app/generate_hdf5.py"),              # image-baked air-gap fallback
    Path("/app/lib/generate_hdf5.py"),
    Path("/app/zarf/scripts/generate_hdf5.py"),
    Path("/root/generate_hdf5.py"),
]
_SCRIPT = next((p for p in _CANDIDATES if p.is_file()), None)
if _SCRIPT is None:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        cand = parent / "zarf" / "scripts" / "generate_hdf5.py"
        if cand.is_file():
            _SCRIPT = cand
            break
if _SCRIPT is None:
    raise FileNotFoundError("generate_hdf5.py not found — mount repo or copy script into image")

sys.path.insert(0, str(_SCRIPT.parent))
import generate_hdf5 as gh

print(f"Using generator: {_SCRIPT}")
print(f"Profiles: {list(gh.PROFILES)}")
for name, p in gh.PROFILES.items():
    print(f"  {name}: {p['description']}")
assert hasattr(gh, "ensure_parts"), (
    "generate_hdf5 is too old (no ensure_parts). "
    "Refresh the sample-notebooks ConfigMap or image copy of generate_hdf5.py."
)


In [ ]:
# Build GenConfig
overrides = {
    "time_unit": TIME_UNIT,
    "contiguous": CONTIGUOUS,
    "group_style": "underscore",
    "product_prefix": True,
    "service_name": "cphy-collector",
    "service_namespace": "cyberphy.otel",
    "base_time": datetime.now(timezone.utc).replace(microsecond=0),
}
if N_PARTS is not None:
    overrides["n_parts"] = int(N_PARTS)
if N_SERIES is not None:
    overrides["n_series"] = int(N_SERIES)
if N_TIME is not None:
    overrides["n_time"] = int(N_TIME)

if PROFILE == "custom":
    cfg = gh.GenConfig(**overrides)
else:
    cfg = gh.profile_config(PROFILE, **overrides)

print("=" * 60)
print(f"Active profile: {PROFILE}")
print(f"  series × time × parts = {cfg.n_series} × {cfg.n_time} × {cfg.n_parts}")
print(f"  Values payload ≈ {cfg.estimated_values_bytes()/1e6:.2f} MB "
      f"({cfg.estimated_values_tib():.4f} TiB)")
print(f"  layout={('contiguous' if cfg.contiguous else 'chunked')}  time_unit={cfg.time_unit}")
print(f"  object keys under s3://{BUCKET}/datasets/hdf5/{cfg.product}/…")
print("=" * 60)
if PROFILE == "lab":
    print("ℹ lab profile is ~10 GiB Values — ensure RUSTFS_DATA_DIR is on /raid.")
if PROFILE == "airgap_2tb":
    print("⚠ airgap_2tb is ~2 TiB — only run on a large air-gap cluster with ample S3.")
    print("  For this notebook session, consider N_PARTS=2 smoke test first.")


## Iceberg / Arrow-oriented layout (dataset plane)

```
s3://cyberphy/
  datasets/hdf5/cphy/                 # Layer A — conserved RO data files
    cphy_<YYYYMMDDTHHMMSS>_<part>Z.h5
  cyberphy-md/iceberg/warehouse/          # Layer B — metadata only (hdf5_iceberg SDK)
    telemetry/hdf5_datasets/parts.parquet # Arrow pointer table
    semantic/catalog.ttl                  # DCAT TTL (not JSON-LD)
```

**Path partitions are not used.** Time bounds and soft tags live in file attrs and the
metadata plane (Arrow / Iceberg). Object keys are flat under the product prefix so
arbitrary customer layouts map through layout adapters without a directory contract.

**Why many small files?** Each file is one fixed-duration acquisition window — a natural
Iceberg *data file* under a future HDF5 `FormatModel`, and a pointer-table row today.

**Contiguous Values:** one byte range per dataset → kerchunk refs degenerate to a single
`(offset, length)`; series hyperslabs are arithmetic strides (`n_time * itemsize` per row).


In [ ]:
# Ensure parts on RustFS — IDEMPOTENT by default (should be seconds, not minutes)
# Look for: "Idempotent reuse: found N … in X.XXs"
# If you see "[1/108] … → s3://…" progress lines, it is GENERATING (slow).
import time as _time

print(f"FORCE_REGENERATE={FORCE_REGENERATE}  OUT={OUT}  endpoint={S3_ENDPOINT or '(default)'}")
if FORCE_REGENERATE:
    print("⚠ FORCE_REGENERATE=True → will WRITE new ~10 GiB lab parts. Set False for reuse.")

_t0 = _time.time()
inventory = gh.ensure_parts(
    cfg,
    OUT,
    s3_endpoint=S3_ENDPOINT or None,
    emit_kerchunk_refs=False,
    dry_run=False,
    progress_every=max(1, cfg.n_parts // 8),
    reuse_existing=not FORCE_REGENERATE,
    force=FORCE_REGENERATE,
)
_elapsed = _time.time() - _t0

pointer_rows = gh.inventory_to_pointer_rows(inventory)
inv_df = pd.DataFrame(pointer_rows)
reused = sum(1 for m in inventory if m.get("reused"))
print(inv_df.head())
print(
    f"\nTotal files: {len(inv_df)}  total size: {inv_df['size_bytes'].sum()/1e6:.1f} MB"
    f"  reused={reused}/{len(inventory)}  wall={_elapsed:.2f}s"
)
if reused == len(inventory) and _elapsed < 5:
    print("✓ Fast path OK — no data rewrite.")
elif reused == 0 and len(inventory):
    print("✗ Generated new parts (not a reuse). Check FORCE_REGENERATE and S3_ENDPOINT.")


In [ ]:
# Pointer-table preview on S3 (small parquet — not the multi-GiB parts)
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.fs as pafs

def s3_filesystem():
    kw = dict(
        access_key=os.environ.get("AWS_ACCESS_KEY_ID", "admin"),
        secret_key=os.environ.get("AWS_SECRET_ACCESS_KEY", "admin"),
        region=S3_REGION,
    )
    token = os.environ.get("AWS_SESSION_TOKEN") or ""
    if token:
        kw["session_token"] = token
    if S3_ENDPOINT:
        kw["endpoint_override"] = S3_ENDPOINT
        kw["scheme"] = "https" if S3_ENDPOINT.startswith("https") else "http"
    return pafs.S3FileSystem(**kw)

s3fs_pa = s3_filesystem()
inv_key = f"{BUCKET}/datasets/hdf5/{cfg.product}/_inventory/parts.parquet"

# Always refresh the small inventory pointer file (cheap); parts themselves are untouched on reuse
table = pa.Table.from_pandas(inv_df)
pq.write_table(table, inv_key, filesystem=s3fs_pa)
print(f"Wrote pointer-table preview → s3://{inv_key}  ({len(inv_df)} rows)")

import s3fs
s3 = s3fs.S3FileSystem(
    key=os.environ.get("AWS_ACCESS_KEY_ID", "admin"),
    secret=os.environ.get("AWS_SECRET_ACCESS_KEY", "admin"),
    client_kwargs={"endpoint_url": S3_ENDPOINT} if S3_ENDPOINT else {},
    config_kwargs={"s3": {"addressing_style": "path"}, "signature_version": "s3v4"},
)
prefix = f"{BUCKET}/datasets/hdf5/{cfg.product}"
print("Sample listing (first 8 .h5):")
for p in sorted(s3.glob(prefix + "/**/*.h5"))[:8]:
    print(" ", p)


## Structural audit (one file)

Open a single acquisition and verify: contiguous Values, uuid chain, time triple agreement,
and Metric sub-window vs acquisition start index.


In [ ]:
import h5py
import tempfile

# Download one part for audit (h5py needs a seekable path for full attr walk)
sample_uri = inventory[0]["uri"]
sample_key = inventory[0]["key"]
print("Auditing", sample_uri)

with tempfile.NamedTemporaryFile(suffix=".h5") as tmp:
    s3.get(f"{BUCKET}/{sample_key}", tmp.name)
    with h5py.File(tmp.name, "r") as f:
        def walk(name, obj):
            kind = "G" if isinstance(obj, h5py.Group) else "D"
            nattr = len(obj.attrs)
            extra = ""
            if isinstance(obj, h5py.Dataset):
                extra = f" shape={obj.shape} dtype={obj.dtype} contiguous={obj.chunks is None}"
            print(f"  {kind} /{name}  attrs={nattr}{extra}")
        print("Tree:")
        f.visititems(walk)
        print("\nRoot uuid:", f.attrs.get("collection.uuid"))
        rm = f["ResourceMetrics"]
        print("Acquisition start.series.index:", int(rm.attrs["start.series.index"]))
        met_name = "Metric_0" if "Metric_0" in rm else "Metric[0]"
        met = rm[met_name]
        print("Metric start.series.index:", int(met.attrs["start.series.index"]))
        print("  (sub-window ≠ acquisition — expected)")
        vals = met["Values"]
        ts = met["Timestamps"]
        print(f"Values: {vals.shape} chunks={vals.chunks} scale.factor={float(met.attrs.get('scale.factor', 0))}")
        print(f"Timestamps: unit={ts.attrs.get('unit')} start.index={int(ts.attrs['start.index'])}")
        print(f"part.start.time Values:", vals.attrs.get("part.start.time"))
        print(f"part.start.time RM:    ", rm.attrs.get("start.time"))


## Dask client + multi-file Values plane

Each object-store part is one acquisition window → one Dask task:

```
S3 part_i.h5  →  worker: h5py Values[::s, ::t]  →  block (n_series', n_time')
                                                      │
                 da.concatenate(..., axis=1)  ─────────┘
                      values_da  shape (series', time' × n_parts)
```

- **No full plane on the client** — only metadata + tiny reductions/viz slabs.
- Workers need S3 reachability (DaskCluster env: `AWS_*`, `S3_ENDPOINT`).
- **Multi-core / multi-node:** scale workers so part tasks run in parallel
  (lab: 4–8, fat node: 8–16, air-gap 2 TiB: 16–32). One worker under-utilizes a
  multi-core box and serializes S3+HDF5 I/O.

Scale via the **DaskCluster CR** (operator source of truth), not a one-off Deployment:

```bash
kubectl -n dask patch daskcluster cybersec-dask --type merge \
  -p '{"spec":{"worker":{"replicas":8}}}'
```

Or at package deploy: `--set DASK_WORKER_REPLICAS=8 --set DASK_WORKER_NTHREADS=2 --set DASK_WORKER_CPU=2`.


In [ ]:
from dask.distributed import Client, get_client, wait
import dask
import dask.array as da
from dask import delayed
import subprocess
import shutil

client = None
if USE_DASK:
    try:
        client = Client(DASK_SCHEDULER, timeout="10s")
        print("Connected:", client)
        print("Dashboard:", client.dashboard_link)
    except Exception as e:
        print(f"Cluster unavailable ({e}); local threaded client")
        client = Client(processes=False, threads_per_worker=2, n_workers=2)
        print(client)
else:
    print("USE_DASK=0 — pure local threads via default scheduler")


def _n_workers(c):
    try:
        return len(c.scheduler_info().get("workers", {}))
    except Exception:
        return 0


def _scale_daskcluster(replicas: int) -> bool:
    """Best-effort scale via kubectl patch of DaskCluster (needs RBAC from notebook)."""
    if not shutil.which("kubectl"):
        return False
    patch = f'{{"spec":{{"worker":{{"replicas":{int(replicas)}}}}}}}'
    cmd = [
        "kubectl", "-n", "dask", "patch", "daskcluster", "cybersec-dask",
        "--type", "merge", "-p", patch,
    ]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if r.returncode == 0:
            print(f"Patched DaskCluster worker.replicas → {replicas}")
            return True
        print(f"kubectl patch failed ({r.returncode}): {(r.stderr or r.stdout).strip()[:200]}")
    except Exception as e:
        print(f"kubectl scale skipped: {e}")
    return False


if client is not None and not str(client).startswith("<Client: inproc"):
    n = _n_workers(client)
    print(f"Workers currently connected: {n}")
    if TARGET_WORKERS and n < TARGET_WORKERS:
        print(
            f"Want ≥{TARGET_WORKERS} workers for multi-core HDF5 (one task/part). "
            f"Attempting scale…"
        )
        scaled = _scale_daskcluster(TARGET_WORKERS)
        if not scaled:
            print(
                "Scale manually (from a host with cluster admin):\n"
                f"  kubectl -n dask patch daskcluster cybersec-dask --type merge "
                f"-p '{{\"spec\":{{\"worker\":{{\"replicas\":{TARGET_WORKERS}}}}}}}'\n"
                "Then re-run this cell."
            )
        try:
            print(f"Waiting up to {WORKER_WAIT_S}s for {TARGET_WORKERS} workers…")
            client.wait_for_workers(n_workers=TARGET_WORKERS, timeout=WORKER_WAIT_S)
        except Exception as e:
            print(f"wait_for_workers: {e} (continuing with {_n_workers(client)} workers)")
    workers = client.scheduler_info().get("workers", {})
    print(f"Workers ready: {len(workers)}")
    total_threads = 0
    for addr, info in list(workers.items())[:12]:
        nt = info.get("nthreads") or info.get("ncores") or 0
        total_threads += int(nt)
        print(
            f"  {addr}  threads={nt}  "
            f"mem_limit={info.get('memory_limit', 0)/1e9:.1f} GiB"
        )
    if len(workers) > 12:
        print(f"  … +{len(workers)-12} more")
    print(f"Aggregate threads: {total_threads}  (parallel part capacity)")
    if len(workers) < 2:
        print(
            "⚠ Only one worker — multi-core box is under-used. "
            "Raise DaskCluster.spec.worker.replicas (see markdown above)."
        )
else:
    if client is not None:
        print("Local client — workers = threads in this process")


def _worker_env_probe():
    import os, socket
    return {
        "host": socket.gethostname(),
        "S3_ENDPOINT": os.environ.get("S3_ENDPOINT") or os.environ.get("AWS_ENDPOINT_URL") or "",
        "AWS_ACCESS_KEY_ID": (os.environ.get("AWS_ACCESS_KEY_ID") or "")[:4] + "…",
        "has_h5py": __import__("importlib").util.find_spec("h5py") is not None,
        "has_s3fs": __import__("importlib").util.find_spec("s3fs") is not None,
    }


if client is not None and _n_workers(client) > 0:
    probe = client.run(_worker_env_probe)
    print("Worker env probe (first 4):")
    for k, v in list(probe.items())[:4]:
        print(f"  {k}: {v}")


In [ ]:
# --- Multi-file Dask array over HDF5 Values (one task per part) ---
# Geometry comes from inventory/cfg — do NOT download a sample on the client.

AK = os.environ.get("AWS_ACCESS_KEY_ID", "admin")
SK = os.environ.get("AWS_SECRET_ACCESS_KEY", "admin")

# Prefer inventory order; cap parts for demos
_meta = inventory[: int(MAX_PARTS)] if MAX_PARTS else list(inventory)
keys = [m["key"] for m in _meta]
n_parts_use = len(keys)
if n_parts_use == 0:
    raise RuntimeError("inventory is empty — run ensure_parts first")

# Declared geometry (reused inventory may only know cfg defaults)
n_series_full = int(_meta[0].get("n_series") or cfg.n_series)
n_time_full = int(_meta[0].get("n_time") or cfg.n_time)
s_stride = max(1, int(SERIES_STRIDE))
t_stride = max(1, int(TIME_STRIDE))
n_series_r = (n_series_full + s_stride - 1) // s_stride
n_time_r = (n_time_full + t_stride - 1) // t_stride
block_shape = (n_series_r, n_time_r)

print(f"Parts in Dask graph: {n_parts_use}")
print(f"Full geometry: {n_series_full} × {n_time_full}  strides: {s_stride}×{t_stride}")
print(f"Block shape per part: {block_shape}  dtype=float32")
print(f"Logical values_da: ({n_series_r}, {n_time_r * n_parts_use})")


def read_values_hyperslab(
    bucket,
    key,
    endpoint,
    access_key,
    secret_key,
    series_stride=1,
    time_stride=1,
):
    """Worker task: fetch one .h5 from S3 and return Values[::ss, ::ts] / scale.

    Runs entirely on a Dask worker (or local scheduler). Returns float32 only —
    never ships the full int16 plane to the notebook unless you .compute() it.
    """
    import tempfile
    import h5py
    import numpy as np
    import s3fs

    s3 = s3fs.S3FileSystem(
        key=access_key,
        secret=secret_key,
        client_kwargs={"endpoint_url": endpoint} if endpoint else {},
        config_kwargs={
            "s3": {"addressing_style": "path"},
            "signature_version": "s3v4",
        },
    )
    with tempfile.NamedTemporaryFile(suffix=".h5") as tmp:
        s3.get(f"{bucket}/{key}", tmp.name)
        with h5py.File(tmp.name, "r") as f:
            rm = f["ResourceMetrics"]
            met = rm["Metric_0"] if "Metric_0" in rm else rm["Metric[0]"]
            ds = met["Values"]
            arr = np.asarray(
                ds[:: max(1, series_stride), :: max(1, time_stride)],
                dtype=np.float32,
            )
            scale = float(met.attrs.get("scale.factor", 1.0)) or 1.0
            return arr / scale


# Build delayed graph — tasks stay lazy until compute/persist
delayed_reads = [
    delayed(read_values_hyperslab)(
        BUCKET, k, S3_ENDPOINT, AK, SK, s_stride, t_stride
    )
    for k in keys
]
blocks = [
    da.from_delayed(d, shape=block_shape, dtype=np.float32) for d in delayed_reads
]
# Concatenate along time: (series, time_total)
values_da = da.concatenate(blocks, axis=1)
values_da = values_da.rechunk({0: -1, 1: n_time_r})  # one chunk per part on time axis

print("Dask array:", values_da)
print("chunks:", values_da.chunks)
print("npartitions (time blocks):", values_da.npartitions)
print(
    "Approx graph bytes if fully realized: "
    f"{values_da.nbytes / 1e9:.3f} GiB  (stays on workers until reduced)"
)

# Optional: show the task graph size (cheap)
print("Delayed layers:", len(values_da.dask))


## Distributed reductions (prove Dask is doing the work)

These cells call `.compute()` on **reductions**, not the full array. Each part is
fetched and reduced on a worker; the client receives scalars / small vectors only.

Open the Dask dashboard while running — you should see tasks proportional to part count.


In [ ]:
# --- Worker-side reductions over all parts ---
import time

# Global mean / std / min / max — tree-reduced across part chunks (workers only)
t0 = time.time()
glob_mean, glob_std, glob_min, glob_max = dask.compute(
    values_da.mean(),
    values_da.std(),
    values_da.min(),
    values_da.max(),
)
print(
    f"global mean={float(glob_mean):.6g}  std={float(glob_std):.6g}  "
    f"min={float(glob_min):.6g}  max={float(glob_max):.6g}  "
    f"({time.time() - t0:.1f}s)"
)

# Per-part mean: each time-chunk is one part (see rechunk above)
def _chunk_mean(block):
    import numpy as np
    return np.asarray([np.nanmean(block)], dtype=np.float64)

# values_da.mean(axis=0) → (time_total,) with n_parts_use chunks
series_mean_t = values_da.mean(axis=0)
part_means = series_mean_t.map_blocks(
    _chunk_mean,
    dtype=np.float64,
    chunks=((1,) * n_parts_use,),
)
t1 = time.time()
part_means_np = np.asarray(part_means.compute(), dtype=np.float64).ravel()
print(
    f"per-part means ({part_means_np.size}): "
    f"{part_means_np[:8]}{'…' if part_means_np.size > 8 else ''}"
)
print(f"per-part reduction wall time: {time.time() - t1:.1f}s")

# Per-series RMS across all time → only n_series_r floats return to client
t2 = time.time()
series_rms = da.sqrt((values_da.astype(np.float64) ** 2).mean(axis=1))
series_rms_np = np.asarray(series_rms.compute(), dtype=np.float64)
print(
    f"series RMS shape={series_rms_np.shape}  "
    f"mean_rms={float(series_rms_np.mean()):.6g}  ({time.time() - t2:.1f}s)"
)
print("↑ Check the Dask dashboard — tasks should scale with part count / graph depth.")


## Interactive heatmap — viewport-driven Dask (lab ≈ multi-TB design)

Static Bokeh zoom on a pre-computed image does **not** touch Dask. For multi-TB that
would be wrong: the client must never hold the plane, and pan/zoom must re-query.

**Design (same code path for lab ~10 GiB and `airgap_2tb`):**

```
visible (series × time) window + screen size
        │
        ▼
  stride so ≈ width × height samples
        │
        ▼
  values_da[s0:s1:ss, t0:t1:ts].compute()   ← only overlapping part-chunks
        │
        ▼
  hv.Image (coords only)  →  dashboard shows new tasks each pan/zoom
```

- One Dask chunk = one object-store part (time axis). Slicing drops non-overlapping parts.
- Series/time strides shrink worker I/O before any bytes hit the notebook.
- Lab dataset is small enough to feel interactive; multi-TB uses the **identical**
  viewport contract (just more parts outside the window).


In [ ]:
# --- Viewport-driven Holoviews: every pan/zoom re-aggregates on Dask ---
import os
import time
import io
import base64

os.environ.setdefault("BOKEH_RESOURCES", "inline")

import holoviews as hv
from holoviews.streams import RangeXY, PlotSize
from bokeh.resources import INLINE
from bokeh.embed import file_html
from IPython.display import display, IFrame, Image as IPImage

hv.config.image_rtol = 1.0
hv.extension("bokeh", inline=True)

try:
    import panel as pn
    pn.extension()
    _HAS_PANEL = True
except Exception as _e:
    _HAS_PANEL = False
    print("panel unavailable:", _e)

# Full logical extents of the multi-file array (already strided at read if SERIES/TIME_STRIDE>1)
_N_SERIES, _N_TIME = int(values_da.shape[0]), int(values_da.shape[1])
_PART_T = int(n_time_r)  # samples per part-chunk on the time axis
print(f"values_da shape (series, time) = ({_N_SERIES}, {_N_TIME})  part_chunk_time={_PART_T}")
print(f"parts in graph: {n_parts_use}  — zoom should schedule only overlapping parts")

# Telemetry for the notebook user (also watch the dashboard)
_viewport_stats = {"calls": 0, "last_s": 0.0, "last_shape": None, "last_parts": 0}


def _array_to_png_bytes(a, width=900, height=420):
    from PIL import Image as PILImage
    a = np.asarray(a, dtype=np.float64)
    finite = np.isfinite(a)
    if finite.any():
        lo, hi = float(a[finite].min()), float(a[finite].max())
        if hi <= lo:
            hi = lo + 1.0
        a = (a - lo) / (hi - lo)
    else:
        a = np.zeros_like(a)
    a = np.clip(a, 0.0, 1.0)
    rgb = (
        np.stack(
            [
                np.clip(1.5 * a, 0, 1),
                np.clip(1.5 * a - 0.5, 0, 1),
                np.clip(1.5 * a - 1.0, 0, 1),
            ],
            axis=-1,
        )
        * 255
    ).astype(np.uint8)
    im = PILImage.fromarray(rgb, mode="RGB").resize((int(width), int(height)), PILImage.NEAREST)
    buf = io.BytesIO()
    im.save(buf, format="PNG")
    return buf.getvalue()


def viewport_to_image(x_range, y_range, width=900, height=420, scale=1.0):
    """Map the visible axes → Dask slice → small Image. Runs on every pan/zoom.

    x_range → time samples, y_range → series indices (matches hv.Image kdims).
    """
    w = max(int(width or 900), 2)
    h = max(int(height or 420), 2)

    if x_range is None or x_range[0] is None:
        t0, t1 = 0, _N_TIME
    else:
        t0 = int(np.floor(min(float(x_range[0]), float(x_range[1]))))
        t1 = int(np.ceil(max(float(x_range[0]), float(x_range[1])))) + 1
    if y_range is None or y_range[0] is None:
        s0, s1 = 0, _N_SERIES
    else:
        s0 = int(np.floor(min(float(y_range[0]), float(y_range[1]))))
        s1 = int(np.ceil(max(float(y_range[0]), float(y_range[1])))) + 1

    t0 = max(0, min(t0, _N_TIME - 1))
    t1 = max(t0 + 1, min(t1, _N_TIME))
    s0 = max(0, min(s0, _N_SERIES - 1))
    s1 = max(s0 + 1, min(s1, _N_SERIES))

    t_span, s_span = t1 - t0, s1 - s0
    # Target ~1 sample per screen pixel (cap work on huge zooms-out)
    t_stride = max(1, int(np.ceil(t_span / w)))
    s_stride = max(1, int(np.ceil(s_span / h)))

    slab = values_da[s0:s1:s_stride, t0:t1:t_stride]
    part0 = t0 // max(_PART_T, 1)
    part1 = (t1 - 1) // max(_PART_T, 1)
    n_parts_touch = int(part1 - part0 + 1)

    t_wall = time.time()
    # *** This is the Dask engagement point for interactive viz ***
    arr = np.asarray(slab.compute(), dtype=np.float32)
    dt = time.time() - t_wall

    ss, tt = arr.shape
    xs = np.arange(t0, t1, t_stride, dtype=np.float64)[:tt]
    ys = np.arange(s0, s1, s_stride, dtype=np.float64)[:ss]
    if arr.shape != (ys.size, xs.size):
        # defensive: align coords to actual array
        ys = ys[: arr.shape[0]]
        xs = xs[: arr.shape[1]]
        arr = arr[: ys.size, : xs.size]

    _viewport_stats["calls"] += 1
    _viewport_stats["last_s"] = dt
    _viewport_stats["last_shape"] = tuple(arr.shape)
    _viewport_stats["last_parts"] = n_parts_touch
    print(
        f"[Dask viewport #{_viewport_stats['calls']}] "
        f"series[{s0}:{s1}:{s_stride}] time[{t0}:{t1}:{t_stride}] "
        f"→ {arr.shape}  parts≈{n_parts_touch}/{n_parts_use}  {dt:.2f}s"
    )

    return hv.Image(
        (xs, ys, arr),
        kdims=["time_sample", "series"],
        vdims=["value"],
        rtol=1.0,
    ).opts(
        cmap="fire",
        colorbar=True,
        width=w,
        height=h,
        xlabel="time sample (concatenated parts)",
        ylabel="series index",
        title=(
            f"CPHY Values — Dask viewport  parts≈{n_parts_touch}/{n_parts_use}  "
            f"{dt:.2f}s  (#{_viewport_stats['calls']})"
        ),
        default_tools=["pan", "wheel_zoom", "box_zoom", "reset", "hover"],
        active_tools=["wheel_zoom"],
        framewise=True,
        axiswise=True,
    )


# Seed full-extent ranges so first paint is a complete overview (still via Dask)
_range = RangeXY(x_range=(0.0, float(_N_TIME)), y_range=(0.0, float(_N_SERIES)))
_size = PlotSize(width=900, height=420)
heatmap_dmap = hv.DynamicMap(viewport_to_image, streams=[_range, _size])
_range.source = heatmap_dmap

print(
    "Pan/zoom the plot below and watch:\n"
    "  • Dask dashboard for new tasks\n"
    "  • stdout lines `[Dask viewport #N] … parts≈k/N`\n"
    "Full multi-TB uses this same windowed contract — only parts under the brush run."
)

if _HAS_PANEL:
    # Panel/ipywidgets comms so DynamicMap callbacks hit the kernel (air-gap, no jupyter_bokeh)
    display(pn.panel(heatmap_dmap, width=940, height=460))
else:
    display(heatmap_dmap)

# Optional: one-shot PNG of the current full overview (no interaction) for export
# overview = viewport_to_image((0, _N_TIME), (0, _N_SERIES), 900, 420)


In [ ]:
# --- Series profile + hist for the *last* Dask viewport (or a fresh mid-series pull) ---
# Re-run after zooming to sample the current window; or pull one series across all time.

def dask_series_profile(series_idx=None, x_range=None):
    """Worker-side 1D profile — only the selected series (and optional time window)."""
    n_s, n_t = int(values_da.shape[0]), int(values_da.shape[1])
    if series_idx is None:
        series_idx = n_s // 2
    series_idx = int(np.clip(series_idx, 0, n_s - 1))
    if x_range is None or x_range[0] is None:
        t0, t1 = 0, n_t
    else:
        t0 = max(0, int(np.floor(min(x_range))))
        t1 = min(n_t, int(np.ceil(max(x_range))) + 1)
    t_span = max(1, t1 - t0)
    t_stride = max(1, t_span // 4000)
    curve_da = values_da[series_idx, t0:t1:t_stride]
    t_wall = time.time()
    y = np.asarray(curve_da.compute(), dtype=np.float32)
    print(
        f"[Dask profile] series={series_idx} time[{t0}:{t1}:{t_stride}] "
        f"→ {y.shape}  {time.time() - t_wall:.2f}s"
    )
    x = np.arange(t0, t1, t_stride, dtype=np.float64)[: y.size]
    return x, y


_s_mid = _N_SERIES // 2
if _viewport_stats.get("calls"):
    print(f"Last viewport: {_viewport_stats}")

x_prof, y_prof = dask_series_profile(
    _s_mid, x_range=_range.x_range if _range.x_range else None
)
profile = hv.Curve((x_prof, y_prof), kdims="time_sample", vdims="value").opts(
    width=900,
    height=200,
    title=f"Series {_s_mid} profile (Dask-fetched)",
    color="#4fc3f7",
)
hist = hv.Histogram(
    np.histogram(y_prof[:: max(1, y_prof.size // 50_000)], bins=80)
).opts(
    width=900,
    height=200,
    title="Value distribution (profile samples)",
    xlabel="normalized phase",
)
layout = (profile + hist).cols(1)

# Prefer Panel (no iframe warning). Fall back to IFrame + INLINE Bokeh for air-gap.
if "_HAS_PANEL" in dir() and _HAS_PANEL:
    display(pn.panel(layout, width=940, height=440))
else:
    try:
        from IPython.display import IFrame

        renderer = hv.renderer("bokeh")
        plot = renderer.get_plot(layout).state
        html_doc = file_html(plot, INLINE, "profile")
        if "cdn.bokeh.org" in html_doc:
            raise RuntimeError("CDN still present in HTML")
        b64 = base64.b64encode(html_doc.encode("utf-8")).decode("ascii")
        display(
            IFrame(
                src=f"data:text/html;base64,{b64}",
                width=940,
                height=480,
            )
        )
    except Exception as e:
        print("profile display fallback:", e)
        display(layout)

if "series_rms_np" in globals():
    top = np.argsort(series_rms_np)[-8:][::-1]
    print("Top series by RMS (from reductions cell):", list(map(int, top)))


## Parallel timestamp continuity (Dask `client.map`)

Each part’s Timestamps bounds are read on workers in parallel — same S3 fetch pattern
as Values, but only returns 4 scalars per file.


In [ ]:
def read_ts_bounds(bucket, key, endpoint, access_key, secret_key):
    import tempfile, h5py, s3fs
    s3 = s3fs.S3FileSystem(
        key=access_key,
        secret=secret_key,
        client_kwargs={"endpoint_url": endpoint} if endpoint else {},
        config_kwargs={"s3": {"addressing_style": "path"}, "signature_version": "s3v4"},
    )
    with tempfile.NamedTemporaryFile(suffix=".h5") as tmp:
        s3.get(f"{bucket}/{key}", tmp.name)
        with h5py.File(tmp.name, "r") as f:
            rm = f["ResourceMetrics"]
            met = rm["Metric_0"] if "Metric_0" in rm else rm["Metric[0]"]
            ts = met["Timestamps"]
            unit = ts.attrs.get("unit", b"ns")
            if isinstance(unit, bytes):
                unit = unit.decode()
            return {
                "key": key,
                "t0": int(ts[0]),
                "t1": int(ts[-1]),
                "start.index": int(ts.attrs["start.index"]),
                "unit": str(unit),
            }


# Parallel map over parts (cap for very large inventories)
_ts_keys = keys[: min(len(keys), 64)]
if client is not None:
    futures = client.map(
        read_ts_bounds,
        [BUCKET] * len(_ts_keys),
        _ts_keys,
        [S3_ENDPOINT] * len(_ts_keys),
        [AK] * len(_ts_keys),
        [SK] * len(_ts_keys),
        pure=False,
    )
    ts_rows = client.gather(futures)
else:
    ts_rows = [
        read_ts_bounds(BUCKET, k, S3_ENDPOINT, AK, SK) for k in _ts_keys
    ]

bdf = pd.DataFrame(ts_rows)
bdf.insert(0, "part", range(len(bdf)))
print(bdf.head(12))
if len(bdf) > 1:
    gaps = bdf["t0"].to_numpy()[1:] - bdf["t1"].to_numpy()[:-1]
    print("inter-part Δ t0[i+1]-t1[i] (first 8):", gaps[:8])


## Next steps (metadata plane)

1. **Registration pipeline** — scan product prefixes / layout adapters → `telemetry.hdf5_datasets` + chunk stats.
2. **Kerchunk** — ranged GETs without full object download (next efficiency win for Dask).
3. **Overviews** — Iceberg parquet pyramids for first-paint datashader.
4. **File Format API** — HDF5 as first-class Iceberg data files.

### Dask workers (multi-core / 2 TiB)

| Environment | Suggested replicas | nthreads | notes |
|-------------|-------------------|----------|--------|
| Laptop / smoke | 1–2 | 2 | `DASK_WORKER_REPLICAS=1` |
| Multi-core lab node (this class) | **8–16** | 2–4 | pack workers; CPU limit ≥ nthreads |
| Air-gap multi-node | **16–32** | 2–4 | ~2–4× node count; spill on fast disk |
| Full `airgap_2tb` (~21.5k parts) | **32+** | 2 | one task/part → want wide pool |

```bash
# Package deploy
zarf package deploy … --set DASK_WORKER_REPLICAS=8 \
  --set DASK_WORKER_NTHREADS=2 --set DASK_WORKER_CPU=2 --set DASK_WORKER_MEMORY=6Gi

# Live scale (operator CR)
kubectl -n dask patch daskcluster cybersec-dask --type merge \
  -p '{"spec":{"worker":{"replicas":8}}}'
```

### Viewport viz (multi-TB contract)

Pan/zoom on the heatmap calls ``values_da[window].compute()`` so **only parts under
the brush** run. Lab ~10 GiB proves the pattern; ``airgap_2tb`` uses the same cell.

### Read knobs

```python
SERIES_STRIDE = 4      # worker-side series decimation
TIME_STRIDE = 2
MAX_PARTS = 16         # limit graph width for demos
TARGET_WORKERS = 8     # notebook waits / tries to scale
```

### Re-runs

```python
FORCE_REGENERATE = False   # default: Run All reuses parts
```


In [ ]:
# Optional cleanup
# if client is not None:
#     client.close()
print("Notebook complete.")
print(f"Parts in inventory: {len(inventory)}  used in Dask graph: {n_parts_use}")
print(f"values_da: {values_da}")
print(f"FORCE_REGENERATE was {FORCE_REGENERATE}")
if client is not None:
    print("Dashboard:", getattr(client, "dashboard_link", ""))
